In [1]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, util
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from transformers import LlamaForCausalLM, LlamaTokenizer

/home/ester/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
candidate_labels = ["Politics", "Religion", "Music", "Gaming", 
"Movies", "Conspiracy", "Finance", "Culture", "Podcast", "Misc"]


In [3]:
df_clean = pd.read_csv("../data/preprocessed_english_titles.csv")
df_clean = df_clean[df_clean["clean_text"].notna()].reset_index(drop=True)


In [6]:
!huggingface-cli login

model_name = "meta-llama/Llama-2-7b-chat-hf" 
tokenizer = LlamaTokenizer.from_pretrained(model_name)
model = LlamaForCausalLM.from_pretrained(model_name, device_map="auto")

⚠️  Warning: 'huggingface-cli login' is deprecated. Use 'hf auth login' instead.

    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): Traceback (most recent call last):
  File "/home/ester/venv/bin/huggingface-cli", line 8, in <module>
    sys.exit(main())
  File "/home/ester/venv/lib/python3.10/site-packages/huggingface_hub/c

OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Llama-2-7b-chat-hf.
401 Client Error. (Request ID: Root=1-68ccbd24-4d0970cf6bc018836932ced1;038891af-e90b-472d-9408-e868d0462835)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-2-7b-chat-hf/resolve/main/tokenizer_config.json.
Access to model meta-llama/Llama-2-7b-chat-hf is restricted. You must have access to it and be authenticated to access it. Please log in.

In [ ]:
def classify_zero_shot(text, candidate_labels):
    prompt = f"Classifique o seguinte título em uma categoria: '{text}'. Categorias: {', '.join(candidate_labels)}. Apenas responda com a categoria correta."
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=20)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    for label in candidate_labels:
        if label.lower() in response.lower():
            return label
    return "Misc"

Carregando o modelo SentenceTransformer
Gerando embeddings para os rótulos...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.58it/s]


In [ ]:
categorias = []
for text in df_clean["clean_text"]: 
    categorias.append(classify_zero_shot(text, candidate_labels))

df_clean["categoria"] = categorias

df_clean.to_csv("../data/preprocessed_english_titles_classified.csv", index=False)

display(df_clean)

Classificando todos os documentos


: 